(SETUP)
Why 5 frames/video: gives ~5,000 real + ~5,000 fake images (10,000 total) — enough to train meaningfully without extraction eating your remaining time.

In [3]:
import cv2
import os
from tqdm import tqdm

real_dest = "D:/Deepfake-detection/datasets/raw_original"
fake_dest = "D:/Deepfake-detection/datasets/raw_deepfakes"

frames_real = "D:/Deepfake-detection/datasets/frames_real"
frames_fake = "D:/Deepfake-detection/datasets/frames_fake"

os.makedirs(frames_real, exist_ok=True)
os.makedirs(frames_fake, exist_ok=True)

FRAMES_PER_VIDEO = 5

(Step 2 - Extraction Function)
What it does: opens each video, picks 5 frame positions evenly spread across its length (not just the first 5 frames, which would look nearly identical), reads and saves each as a .jpg.

In [4]:
def extract_frames(video_path, output_folder, video_id, num_frames=5):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames <= 0:
        cap.release()
        return 0
    
    frame_indices = [int(i * total_frames / num_frames) for i in range(num_frames)]
    
    saved = 0
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, frame = cap.read()
        if success:
            out_path = os.path.join(output_folder, f"{video_id}_frame{saved}.jpg")
            cv2.imwrite(out_path, frame)
            saved += 1
    
    cap.release()
    return saved

Step 3- Extract real video Frames

In [8]:
real_videos = os.listdir(real_dest)

total_saved = 0
for i, vid in enumerate(tqdm(real_videos, desc="Extracting real frames")):
    video_path = os.path.join(real_dest, vid)
    saved = extract_frames(video_path, frames_real, video_id=f"real_{i}", num_frames=FRAMES_PER_VIDEO)
    total_saved += saved

print(f"Total real frames saved: {total_saved}")

Extracting real frames: 100%|██████████| 1000/1000 [20:53<00:00,  1.25s/it]

Total real frames saved: 5000


Step 4- Extract fake video frames

In [5]:
fake_videos = os.listdir(fake_dest)

total_saved = 0
for i, vid in enumerate(tqdm(fake_videos, desc="Extracting fake frames")):
    video_path = os.path.join(fake_dest, vid)
    saved = extract_frames(video_path, frames_fake, video_id=f"fake_{i}", num_frames=FRAMES_PER_VIDEO)
    total_saved += saved

print(f"Total fake frames saved: {total_saved}")

Extracting fake frames: 100%|██████████| 1000/1000 [19:52<00:00,  1.19s/it]

Total fake frames saved: 5000


Step 5 - Install a face detector (fast, lightweight)

In [6]:
!pip install mtcnn

  Using cached mtcnn-1.0.0-py3-none-any.whl.metadata (5.8 kB)
Using cached mtcnn-1.0.0-py3-none-any.whl (1.9 MB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [mtcnn]
   -------------------------- --

Why MTCNN specifically: it's a well-established, reasonably fast face-detection model, easy to use in 2-3 lines, and doesn't require GPU to run reasonably (important since we want GPU free for training, not face-detection preprocessing).

In [11]:
!pip uninstall opencv-python opencv-python-headless -y
!pip install opencv-python==4.10.0.84

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93


You can safely remove it manually.


   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.8 MB ? eta -:--:--
    --------------------------------------- 0.5/38.8 MB 409.0 kB/s eta 0:01:34
    --------------------------------------- 0.5/38.8 MB 409.0 kB/s eta 0:01:34
    --------------------------------------- 0.8/38.8 MB 479.2 kB/s eta 0:01:20
    --------------------------------------- 0.8/38.8 MB 479.2 kB/s eta 0:01:20
    --------------------------------------- 0.8/38.8 MB 479.2 kB/s eta 0:01:20
   - -------------------------------------- 1.0/38.8 MB 445.3 kB/s eta 0:01:25
   - -------------------------------------- 1.0/38.8 MB 445.3 kB/s eta 0:01:25
   - -------------------------------------- 1.0/38.8 MB 445.3 kB/s eta 0:01:25
   - -------------

In [12]:
!pip install opencv-python==4.10.0.84

In [1]:
import cv2
import os
from tqdm import tqdm

real_dest = "D:/Deepfake-detection/datasets/raw_original"
fake_dest = "D:/Deepfake-detection/datasets/raw_deepfakes"

frames_real = "D:/Deepfake-detection/datasets/frames_real"
frames_fake = "D:/Deepfake-detection/datasets/frames_fake"

print(cv2.__version__)

4.10.0


In [6]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
print("Cascade loaded:", not face_cascade.empty())

Cascade loaded: True


Step 6 - Face crop + resize function

In [1]:
IMG_SIZE = 224

def crop_face(image_path, output_path, img_size=IMG_SIZE):
    img = cv2.imread(image_path)
    if img is None:
        return "unreadable"
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    
    if len(faces) > 0:
        # Face found — crop to the largest detected face
        faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
        x, y, w, h = faces[0]
        face_crop = img[y:y+h, x:x+w]
        method = "face_crop"
    else:
        # No face found — fall back to using the FULL frame instead of dropping it
        face_crop = img
        method = "full_frame"
    
    face_resized = cv2.resize(face_crop, (img_size, img_size))
    cv2.imwrite(output_path, face_resized)
    return method

Step 7 - Processing loop- tracks breakdown

In [7]:
cropped_real = "D:/Deepfake-detection/datasets/cropped_real"
cropped_fake = "D:/Deepfake-detection/datasets/cropped_fake"
os.makedirs(cropped_real, exist_ok=True)
os.makedirs(cropped_fake, exist_ok=True)

def process_folder(input_folder, output_folder):
    files = os.listdir(input_folder)
    counts = {"face_crop": 0, "full_frame": 0, "unreadable": 0}
    
    for f in tqdm(files, desc=f"Processing {input_folder}"):
        in_path = os.path.join(input_folder, f)
        out_path = os.path.join(output_folder, f)
        result = crop_face(in_path, out_path)
        counts[result] += 1
    
    return counts

real_counts = process_folder(frames_real, cropped_real)
print(f"Real: {real_counts}, total processed: {sum(real_counts.values())} / {len(os.listdir(frames_real))}")

fake_counts = process_folder(frames_fake, cropped_fake)
print(f"Fake: {fake_counts}, total processed: {sum(fake_counts.values())} / {len(os.listdir(frames_fake))}")

Processing D:/Deepfake-detection/datasets/frames_real: 100%|██████████| 5000/5000 [09:40<00:00,  8.61it/s]


Real: {'face_crop': 4973, 'full_frame': 27, 'unreadable': 0}, total processed: 5000 / 5000


Processing D:/Deepfake-detection/datasets/frames_fake: 100%|██████████| 5000/5000 [10:23<00:00,  8.01it/s]

Fake: {'face_crop': 4949, 'full_frame': 51, 'unreadable': 0}, total processed: 5000 / 5000


Step 8 - Train/Validation/Test split


random.shuffle(files) — randomizes order before splitting, so we don't accidentally put all of one video's frames only into train (important since consecutive frames from the same video look very similar — shuffling reduces bias)
train_ratio=0.7, val_ratio=0.15 → automatically gives 70% train / 15% validation / 15% test (matching the split we discussed back in Phase 6)
Creates the proper folder structure your Phase 3 plan expects: datasets/train/real, datasets/train/fake, datasets/validation/real, etc.
shutil.copy — copies (not moves) files, so your original cropped_real/cropped_fake folders stay intact as a backup

Result: roughly 3,500 train / 750 val / 750 test per class (real and fake), landing in the exact folder structure PyTorch's ImageFolder loader expects — which is exactly what we'll use in the next step to build the model.

In [9]:
import shutil
import random

random.seed(42)

def split_dataset(source_folder, dest_base, class_name, train_ratio=0.7, val_ratio=0.15):
    files = os.listdir(source_folder)
    random.shuffle(files)
    
    n = len(files)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)
    
    train_files = files[:train_end]
    val_files = files[train_end:val_end]
    test_files = files[val_end:]
    
    for split_name, split_files in [("train", train_files), ("validation", val_files), ("test", test_files)]:
        dest_folder = os.path.join(dest_base, split_name, class_name)
        os.makedirs(dest_folder, exist_ok=True)
        for f in split_files:
            shutil.copy(os.path.join(source_folder, f), os.path.join(dest_folder, f))
    
    print(f"{class_name}: train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

dataset_base = "D:/Deepfake-detection/datasets"

split_dataset(cropped_real, dataset_base, "real")
split_dataset(cropped_fake, dataset_base, "fake")

real: train=3500, val=750, test=750
fake: train=3500, val=750, test=750
